<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Comparative Analysis of RNN and LSTM Architectures</h1> 


Estimated time needed: **25** minutes


## Learning Objectives 


In this lab, you will learn how to:
- Implement a vanilla RNN and an LSTM network in PyTorch on the same sequential dataset
- Compare training dynamics, gradient behavior, and convergence speed between both architectures
- Evaluate and interpret performance differences with AI-assisted support to justify when LSTM outperforms RNN 


<h2>Table of Contents</h2>
<p>In this lab, you will train a vanilla RNN and an LSTM on the same sequence classification dataset and compare their behavior.</p>

<ul>
<ol>
<li><a href="#RNN-vs-LSTM-Architecture">RNN vs LSTM Architecture</a> 
</li>
<li><a href="#Create-a-Sequential-Dataset">Create a Sequential Dataset</a>
</li> 
 <li><a href="#Preprocess-and-Batch-Text">Preprocess and Batch Text</a>
</li> 
 <li><a href="#Implement-RNN-and-LSTM-Models">Implement RNN and LSTM Models</a>
</li> 
 <li><a href="#Train-both-Architectures">Train both Architectures</a>
</li>  
 <li><a href="#Compare-Gradient-Behavior">Compare-Gradient-Behavior</a>
</li> 
<li><a href="#Compare-Convergence-Speed">Compare Convergence Speed</a>
</li> 
<li><a href="#AI-assisted-Model-Comparison">AI-assisted Model Comparison</a>
</li> 

</li> 
</ol>
</ul>


<h2>Import the Required Libraries</h2>

Use the code below to import the libraries


In [1]:
# Install PyTorch

!pip install torch

In [2]:
# Import the libraries we need for this lab

import random
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader, random_split

random.seed(2)
torch.manual_seed(2)

/opt/conda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))

<!--Empty Space for separating topics-->


## RNN vs LSTM Architecture


A vanilla RNN updates a hidden state at each time step. An LSTM adds gates and a cell state, which helps preserve useful information across longer sequences.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Recurrent%20neural%20network%20unfold.svg" width="650" alt="Unfolded recurrent neural network diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Recurrent neural network unfold.svg</em></p>


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Long%20Short-Term%20Memory.svg" width="650" alt="Long short-term memory architecture diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Long Short-Term Memory.svg</em></p>


This lab compares models using the same data, vocabulary, train-validation split, loss function, and optimizer so the architectural differences are easier to observe.


<!--Empty Space for separating topics-->


## Create a Sequential Dataset


Create a sentiment dataset with positive and negative text sequences. The label <code>1</code> represents positive sentiment, and the label <code>0</code> represents negative sentiment.


In [3]:
# Create a sentiment dataset

positive_sentences = [
    "the product works well and remains reliable after daily use",
    "the support team solved the issue quickly and politely",
    "the course examples were clear practical and easy to follow",
    "the app opens fast responds smoothly and saves my work",
    "the package arrived early with every item protected",
    "the headphones stay connected and sound balanced for hours",
    "the dashboard loads quickly and explains the results clearly",
    "the device feels durable even after repeated travel use",
    "the lesson sequence builds ideas carefully from start to finish",
    "the service experience was friendly efficient and accurate",
]

negative_sentences = [
    "the product fails often and becomes unreliable after daily use",
    "the support team ignored the issue repeatedly and rudely",
    "the course examples were confusing outdated and hard to follow",
    "the app opens slowly freezes often and loses my work",
    "the package arrived late with several items damaged",
    "the headphones disconnect constantly and sound distorted for hours",
    "the dashboard loads slowly and hides the results badly",
    "the device feels fragile even after careful indoor use",
    "the lesson sequence skips ideas abruptly from start to finish",
    "the service experience was rude inefficient and inaccurate",
]

base_data = [(sentence, 1) for sentence in positive_sentences] + [(sentence, 0) for sentence in negative_sentences]
random.shuffle(base_data)

print("Base examples:", len(base_data))
print(base_data[:3])

Base examples: 20
[('the device feels durable even after repeated travel use', 1), ('the dashboard loads quickly and explains the results clearly', 1), ('the device feels fragile even after careful indoor use', 0)]

Expand the dataset using short context prefixes. This creates more sequence examples while preserving class labels.


In [4]:
# Expand the dataset

prefixes = ["", "overall ", "after one week ", "in my experience ", "for a long task "]
expanded_data = []

for sentence, label in base_data:
    for prefix in prefixes:
        expanded_data.append((prefix + sentence, label))

random.shuffle(expanded_data)

print("Expanded examples:", len(expanded_data))
print(expanded_data[:3])

Expanded examples: 100
[('the dashboard loads quickly and explains the results clearly', 1), ('after one week the device feels fragile even after careful indoor use', 0), ('in my experience the product works well and remains reliable after daily use', 1)]

<!--Empty Space for separating topics-->


## Preprocess and Batch Text


Tokenize each sentence, build a vocabulary, and encode each sequence as token IDs.


In [5]:
# Tokenize text

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_examples = [(tokenize(text), label) for text, label in expanded_data]

print(tokenized_examples[0])

(['the', 'dashboard', 'loads', 'quickly', 'and', 'explains', 'the', 'results', 'clearly'], 1)

Build a vocabulary with padding and unknown tokens:


In [6]:
# Build vocabulary

token_counts = Counter(token for tokens, label in tokenized_examples for token in tokens)
special_tokens = ["<pad>", "<unk>"]
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))

Vocabulary size: 110

Encode tokens and create a PyTorch Dataset:


In [7]:
# Encode tokens and create a Dataset

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

encoded_examples = [(encode_tokens(tokens, vocab), label) for tokens, label in tokenized_examples]

class SequenceDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        token_ids, label = self.examples[index]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float32)


full_dataset = SequenceDataset(encoded_examples)
print("Dataset size:", len(full_dataset))

Dataset size: 100

Split the dataset and create padded batches:


In [8]:
# Create train and validation DataLoaders

train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(2),
)

def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    labels = torch.stack(labels)
    return padded_sequences, labels, lengths

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

batch_sequences, batch_labels, batch_lengths = next(iter(train_loader))
print(batch_sequences.shape, batch_labels.shape, batch_lengths)

torch.Size([8, 14]) torch.Size([8]) tensor([12, 13, 13, 12, 10, 14, 10, 12])

<!--Empty Space for separating topics-->


## Implement RNN and LSTM Models


Define two classifiers with the same embedding size, hidden size, and output layer shape. The only difference is the recurrent layer type.


In [9]:
# Define a vanilla RNN classifier

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed)
        return self.fc(hidden[-1]).squeeze(1)

In [10]:
# Define an LSTM classifier

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed)
        return self.fc(hidden[-1]).squeeze(1)

Create helper functions to count parameters and compute gradient norms:


In [11]:
# Create parameter and gradient helpers

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def gradient_norm(model):
    total_norm = 0
    for parameter in model.parameters():
        if parameter.grad is not None:
            total_norm += parameter.grad.detach().data.norm(2).item() ** 2
    return total_norm ** 0.5

<!--Empty Space for separating topics-->


## Train both Architectures


Use the same loss function, optimizer type, learning rate, and number of epochs for both models.


In [12]:
# Create training and evaluation functions

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for sequences, labels, lengths in loader:
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return total_loss / len(loader), correct / total

def train_model(model, train_loader, valid_loader, epochs=8):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    history = []
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        grad_norms = []
        for sequences, labels, lengths in train_loader:
            optimizer.zero_grad()
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            grad_norms.append(gradient_norm(model))
            optimizer.step()
            train_loss += loss.item()
        valid_loss, valid_accuracy = evaluate_model(model, valid_loader, criterion)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss / len(train_loader),
            "valid_loss": valid_loss,
            "valid_accuracy": valid_accuracy,
            "avg_grad_norm": sum(grad_norms) / len(grad_norms),
        })
    return history

Instantiate and train both models:


In [13]:
# Train RNN and LSTM models

torch.manual_seed(2)
rnn_model = RNNClassifier(len(vocab), embedding_dim=32, hidden_dim=32, padding_idx=pad_id)

torch.manual_seed(2)
lstm_model = LSTMClassifier(len(vocab), embedding_dim=32, hidden_dim=32, padding_idx=pad_id)

print("RNN parameters:", count_parameters(rnn_model))
print("LSTM parameters:", count_parameters(lstm_model))

rnn_history = train_model(rnn_model, train_loader, valid_loader, epochs=8)
lstm_history = train_model(lstm_model, train_loader, valid_loader, epochs=8)

RNN parameters: 5665
LSTM parameters: 12001

Print training histories for both architectures:


In [14]:
# Display model histories

for rnn_row, lstm_row in zip(rnn_history, lstm_history):
    print("Epoch", rnn_row["epoch"])
    print("RNN :", rnn_row)
    print("LSTM:", lstm_row)

Epoch 1
RNN : {'epoch': 1, 'train_loss': 0.6587823390960693, 'valid_loss': 0.5075660149256388, 'valid_accuracy': 0.8, 'avg_grad_norm': 0.8583182513363884}
LSTM: {'epoch': 1, 'train_loss': 0.6605018079280853, 'valid_loss': 0.5854389667510986, 'valid_accuracy': 0.8, 'avg_grad_norm': 0.29371158232573824}
Epoch 2
RNN : {'epoch': 2, 'train_loss': 0.35923137664794924, 'valid_loss': 0.2876904308795929, 'valid_accuracy': 0.9, 'avg_grad_norm': 0.7275772044448939}
LSTM: {'epoch': 2, 'train_loss': 0.3897522360086441, 'valid_loss': 0.25656140844027203, 'valid_accuracy': 0.95, 'avg_grad_norm': 0.42554700031903286}
Epoch 3
RNN : {'epoch': 3, 'train_loss': 0.11227887347340584, 'valid_loss': 0.037341877818107605, 'valid_accuracy': 1.0, 'avg_grad_norm': 0.5224336838704015}
LSTM: {'epoch': 3, 'train_loss': 0.06642573978751898, 'valid_loss': 0.017731713131070137, 'valid_accuracy': 1.0, 'avg_grad_norm': 0.28234110298590165}
Epoch 4
RNN : {'epoch': 4, 'train_loss': 0.021115761157125233, 'valid_loss': 0.010

<!--Empty Space for separating topics-->


## Compare Gradient Behavior


Gradient norms provide a simple way to inspect training stability. Very small gradients may suggest vanishing gradients, while very large gradients may suggest instability.


In [15]:
# Compare gradient norms

for rnn_row, lstm_row in zip(rnn_history, lstm_history):
    print(
        "Epoch", rnn_row["epoch"],
        "RNN grad norm=", round(rnn_row["avg_grad_norm"], 4),
        "LSTM grad norm=", round(lstm_row["avg_grad_norm"], 4),
    )

Epoch 1 RNN grad norm= 0.8583 LSTM grad norm= 0.2937
Epoch 2 RNN grad norm= 0.7276 LSTM grad norm= 0.4255
Epoch 3 RNN grad norm= 0.5224 LSTM grad norm= 0.2823
Epoch 4 RNN grad norm= 0.1103 LSTM grad norm= 0.2506
Epoch 5 RNN grad norm= 0.0312 LSTM grad norm= 0.0858
Epoch 6 RNN grad norm= 0.0168 LSTM grad norm= 0.0373
Epoch 7 RNN grad norm= 0.0122 LSTM grad norm= 0.0114
Epoch 8 RNN grad norm= 0.0097 LSTM grad norm= 0.0067

<!--Empty Space for separating topics-->


<h2 id="Convergence">Compare Convergence Speed</h2>


Convergence speed can be estimated by checking when validation loss first drops below a target threshold.


In [16]:
# Compare convergence speed

def first_epoch_below(history, threshold):
    for row in history:
        if row["valid_loss"] <= threshold:
            return row["epoch"]
    return None

threshold = 0.4
rnn_convergence_epoch = first_epoch_below(rnn_history, threshold)
lstm_convergence_epoch = first_epoch_below(lstm_history, threshold)

print("RNN convergence epoch:", rnn_convergence_epoch)
print("LSTM convergence epoch:", lstm_convergence_epoch)

RNN convergence epoch: 2
LSTM convergence epoch: 2

Compare final validation performance:


In [17]:
# Compare final metrics

final_comparison = {
    "RNN": rnn_history[-1],
    "LSTM": lstm_history[-1],
}

final_comparison

{'RNN': {'epoch': 8,
  'train_loss': 0.001967636775225401,
  'valid_loss': 0.0020089720686276755,
  'valid_accuracy': 1.0,
  'avg_grad_norm': 0.009680456379249351},
 'LSTM': {'epoch': 8,
  'train_loss': 0.0017140317708253861,
  'valid_loss': 0.001364310543673734,
  'valid_accuracy': 1.0,
  'avg_grad_norm': 0.00673549854679506}}

<!--Empty Space for separating topics-->


## AI-assisted Model Comparison


AI-assisted support can help summarize training trends, but the conclusion should be based on observed loss, accuracy, gradient norms, and convergence behavior.


In [18]:
# Create a rule-based comparison summary

def compare_models(rnn_history, lstm_history):
    rnn_final = rnn_history[-1]
    lstm_final = lstm_history[-1]

    if lstm_final["valid_accuracy"] > rnn_final["valid_accuracy"]:
        accuracy_summary = "The LSTM achieved higher validation accuracy."
    elif lstm_final["valid_accuracy"] == rnn_final["valid_accuracy"]:
        accuracy_summary = "Both models achieved the same validation accuracy."
    else:
        accuracy_summary = "The RNN achieved higher validation accuracy on this small dataset."

    if lstm_final["valid_loss"] < rnn_final["valid_loss"]:
        loss_summary = "The LSTM ended with lower validation loss."
    else:
        loss_summary = "The RNN ended with lower or equal validation loss."

    return accuracy_summary + " " + loss_summary + " LSTMs are usually preferred when longer dependencies matter because gates and cell state help preserve sequence information."

print(compare_models(rnn_history, lstm_history))

Both models achieved the same validation accuracy. The LSTM ended with lower validation loss. LSTMs are usually preferred when longer dependencies matter because gates and cell state help preserve sequence information.

If the dataset is short and simple, a vanilla RNN may perform similarly. LSTM advantages usually become clearer on longer sequences, noisier text, and tasks requiring information from earlier time steps.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--Empty Space for separating topics-->


<h2>About the Author:</h2> 

<b> Harish Pant </b> is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations. 


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-11  | 1.0  | 029  |  Created lab for comparative analysis of RNN and LSTM architectures |
-->





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
